# Experiment Tracking: QLoRA Fine-Tuning Mistral-7B

## Previous runs (from other accounts)

| Run | Epochs | LR | Rank | Best Val Loss | Best Step | Status | Key Finding |
|-----|--------|----|------|---------------|-----------|--------|-------------|
| Run 1 | 3 | 2e-4 | 16 | 1.475 | 200 | Overfit | Val loss rose to 1.903 by step 725. Model memorized training data. |
| Run 2 | 1 | 1e-4 | 16 | 1.592 | 50 | Underfit | Val loss flat at 1.592 for all steps. LR too low, too few epochs. |
| Run 3 | 2 | 2e-4 | 16 | 1.422 | 200 | Good | Healthy curve. Val loss improved until step 200, mild overfit after. |
| Run 4 | 2 | 1.5e-4 | 16 | 1.419 | 200 | Best | Lowest val loss. Slightly better than Run 3. Adapters lost in crash. |
| Run 5 | 2 | 2e-4 | 32 | - | - | Never ran | Session crashed before this experiment. |

## What we learned
- 3 epochs overfits badly on this dataset size (~1800 unique examples)
- 1 epoch with lr=1e-4 is too conservative — model barely learns
- 2 epochs is the sweet spot — enough learning without severe overfitting
- lr=1.5e-4 slightly better than lr=2e-4 (1.419 vs 1.422)
- Early stopping with patience=3 catches the overfit correctly
- Best model always at step ~200 regardless of LR

## Remaining experiments
- Run 4 retrain (winner): 2 epochs, lr=1.5e-4, rank=16
- Run 5 (new): 2 epochs, lr=2e-4, rank=32 — test if higher rank helps

## All runs use
- Model: unsloth/mistral-7b-instruct-v0.3-bnb-4bit
- Quantization: 4-bit NF4
- LoRA targets: q, k, v, o, gate, up, down projections
- Optimizer: adamw_8bit
- Scheduler: cosine with 30 warmup steps
- Batch size: 2 x 4 gradient accumulation = 8 effective
- Eval every 50 steps
- Early stopping patience: 3

In [1]:
# Verifying if GPU is available

import torch

print(f"GPU avaliable: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU avaliable: True
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
# Install dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes triton
!pip install mlflow==2.19.0

import torch
print(f"GPU avaliable: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7p73oyhh/unsloth_773377060247440289d1a46adc7ac37d
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7p73oyhh/unsloth_773377060247440289d1a46adc7ac37d
  Resolved https://github.com/unslothai/unsloth.git to commit 5c473fab80e079bb525345b86cb71afd409262c3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl (48.9 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following depe

In [5]:
!pip install pyngrok

In [6]:
# Data loading
from google.colab import drive
import json
from datasets import Dataset
# Model loading
from unsloth import FastLanguageModel
import torch
# Importing training components
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers import EarlyStoppingCallback
# Training environment setup
import mlflow
import os

import subprocess
import time
from pyngrok import ngrok

In [ ]:
# Load data and MLflow
from google.colab import drive
drive.mount('/content/drive')

import json, os, time, torch, mlflow
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback

save_dir = "/content/drive/MyDrive/review-summarizer"

train_dataset = Dataset.from_json(f"{save_dir}/train_dataset_sft.jsonl")
val_dataset = Dataset.from_json(f"{save_dir}/val_dataset_sft.jsonl")
with open(f"{save_dir}/test_data.json") as f:
    test_data = json.load(f)

os.system(f"cp {save_dir}/mlflow_fresh.db /tmp/mlflow_fresh.db")
mlflow.set_tracking_uri("sqlite:////tmp/mlflow_fresh.db")
mlflow.set_experiment("product-review-summarizer")

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_data)}")
print(f"Existing MLflow runs: {len(mlflow.search_runs())}")

Mounted at /content/drive


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train: 1800, Val: 186, Test: 187
Existing MLflow runs: 4


1. max_seq_length=2048 --- the longest sequence the model will handle during training. Our reviews + summaries are typically 300-500 tokens, so 2048 gives plenty of headroom. Higher values use more memory.
2. dtype=None --- lets Unsloth auto-detect the best precision for your GPU. T4 supports float16, A100 supports bfloat16. bfloat16 is more numerically stable but T4 doesn't have it. Unsloth handles this for you.
3. load_in_4bit=True --- this is the "Q" in QLoRA. The frozen base weights get quantized to 4-bit NF4 format, cutting memory from 14 GB to 3.5 GB.


1. The model was over fitting - training loss was decreasing and the eval loss was increasing.
2. Added early stopping
3. Lowered learning rate to 1e-4 from 2e-4. Half of what we used before. lowering learning rate - slowers learning, less overfitting

In [10]:
# Define run_experiment (with immediate Drive saves)
def run_experiment(config):
    print(f"\n{'='*60}")
    print(f"EXPERIMENT: {config['name']}")
    print(f"Epochs={config['epochs']}, LR={config['lr']}, Rank={config['rank']}")
    print(f"{'='*60}\n")

    os.environ["MLFLOW_RUN_NAME"] = config["name"]

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
        max_seq_length=2048, dtype=None, load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model, r=config["rank"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_alpha=config["rank"],
        lora_dropout=0, bias="none",
        use_gradient_checkpointing="unsloth", random_state=42,
    )
    model.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=f"./results_{config['name']}",
        num_train_epochs=config["epochs"],
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=config["lr"],
        lr_scheduler_type="cosine",
        warmup_steps=30,
        weight_decay=0.01,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        seed=42,
        report_to="mlflow",
    )

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=train_dataset, eval_dataset=val_dataset,
        dataset_text_field="text", max_seq_length=2048,
        packing=False, args=training_args,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    start_time = time.time()
    stats = trainer.train()
    train_time = time.time() - start_time

    # SAVE EVERYTHING TO DRIVE IMMEDIATELY
    adapter_dir = f"./adapters_{config['name']}"
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    os.system(f"cp -r {adapter_dir} {save_dir}/adapters_{config['name']}")
    print(f"Adapters saved to Drive!")

    os.system(f"cp /tmp/mlflow_fresh.db {save_dir}/mlflow_fresh.db")
    print(f"MLflow DB saved to Drive!")

    best_ckpt = trainer.state.best_model_checkpoint
    if best_ckpt and os.path.exists(best_ckpt):
        os.system(f"cp -r {best_ckpt} {save_dir}/best_checkpoint_{config['name']}")
        print(f"Best checkpoint saved to Drive!")

    runs = mlflow.search_runs(
        experiment_names=["product-review-summarizer"],
        order_by=["start_time DESC"],
    )
    with mlflow.start_run(run_id=runs.iloc[0]["run_id"]):
        mlflow.log_params({
            "lora_rank": config["rank"],
            "lora_alpha": config["rank"],
            "quantization": "4bit-NF4",
            "description": config["description"],
        })
        mlflow.log_metric("training_time_minutes", train_time / 60)

    os.system(f"cp /tmp/mlflow_fresh.db {save_dir}/mlflow_fresh.db")

    response = test_model(model, tokenizer)

    print(f"\nTrain loss: {stats.training_loss:.4f}")
    print(f"Best checkpoint: {best_ckpt}")
    print(f"Time: {train_time/60:.1f} min")
    print(f"\nSample output:\n{response}")

    del model, tokenizer, trainer
    torch.cuda.empty_cache()

    return stats

print("Function defined!")

Function defined!


In [9]:
# Define test function
test_review = """This tablet is decent for the price. The screen is bright
and colorful, and the battery easily lasts 8 hours. However, the speakers
are really tinny, and it gets noticeably slow when you have more than
3 apps open. The camera is basically useless in low light."""

def test_model(model, tokenizer):
    FastLanguageModel.for_inference(model)
    prompt = f"""<s>[INST] You are a product review analyzer. Given a customer review, generate a structured summary with pros, cons, and an overall verdict.

Review: {test_review}
Rating: 3/5 [/INST]
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs, max_new_tokens=256,
        temperature=0.7, top_p=0.9, repetition_penalty=1.1,
    )
    return tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )

In [ ]:
# Run 1 + rank=16 experiment
results_1 = run_experiment({
    "name": "run1-3ep-lr2e4-r16",
    "epochs": 2,
    "lr": 1.5e-4,
    "rank": 16,
    "description": "Winner retrain: 2 epochs, lr=1.5e-4, rank=16"
})

In [ ]:
# Run 2 + rank=16 experiment
results_1 = run_experiment({
    "name": "run2-1ep-lr1e4-r16",
    "epochs": 2,
    "lr": 1.5e-4,
    "rank": 16,
    "description": "Winner retrain: 2 epochs, lr=1.5e-4, rank=16"
})

In [ ]:
# Run 3 + rank=16 experiment
results_1 = run_experiment({
    "name": "run3-2ep-lr2e4-r16",
    "epochs": 2,
    "lr": 1.5e-4,
    "rank": 16,
    "description": "Winner retrain: 2 epochs, lr=1.5e-4, rank=16"
})

In [11]:
# Run the winning config (Run 4) + rank=16 experiment
results_4 = run_experiment({
    "name": "run4-2ep-lr1.5e4-r16",
    "epochs": 2,
    "lr": 1.5e-4,
    "rank": 16,
    "description": "Winner retrain: 2 epochs, lr=1.5e-4, rank=16"
})


EXPERIMENT: run4-2ep-lr1.5e4-r16
Epochs=2, LR=0.00015, Rank=16

==((====))==  Unsloth 2026.4.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Adapters saved to Drive!
MLflow DB saved to Drive!


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Best checkpoint saved to Drive!


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


Train loss: 1.2657
Best checkpoint: ./results_run4-2ep-lr1.5e4-r16/checkpoint-200
Time: 33.8 min

Sample output:
**Pros:**
- Bright and colorful screen
- Battery lasts 8 hours

**Cons:**
- Tinny speakers
- Gets slow with multiple apps
- Camera is useless in low light

**Verdict:** Overall, it's a decent tablet for the price but has significant drawbacks.


In [12]:
# Run rank=32 experiment
results_5 = run_experiment({
    "name": "run5-2ep-lr2e4-r32",
    "epochs": 2,
    "lr": 2e-4,
    "rank": 32,
    "description": "Higher rank: more model capacity"
})


EXPERIMENT: run5-2ep-lr2e4-r32
Epochs=2, LR=0.0002, Rank=32

==((====))==  Unsloth 2026.4.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

trainable params: 83,886,080 || all params: 7,331,909,632 || trainable%: 1.1441


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/1800 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/186 [00:00<?, ? examples/s]

...

Adapters saved to Drive!
MLflow DB saved to Drive!


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Best checkpoint saved to Drive!


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



Train loss: 1.1739
Best checkpoint: ./results_run5-2ep-lr2e4-r32/checkpoint-200
Time: 34.5 min

Sample output:
**Pros:**
- Bright and colorful screen
- Battery lasts 8 hours

**Cons:**
- Tinny speakers
- Camera is useless in low light
- Gets slow with multiple apps

**Verdict:** Overall, it's a decent tablet for the price.


In [13]:
# Verify everything is on Drive
print("Files on Drive:")
!ls -lh {save_dir}/
print("\nAdapters:")
!ls -d {save_dir}/adapters_* 2>/dev/null || echo "No adapters found"

Files on Drive:
total 4.7M
drwx------ 2 root root 4.0K Apr 24 01:06 adapters_run4-2ep-lr1.5e4-r16
drwx------ 2 root root 4.0K Apr 24 01:45 adapters_run5-2ep-lr2e4-r32
-rw------- 1 root root 775K Apr 18 07:25 balanced_reviews.parquet
drwx------ 2 root root 4.0K Apr 24 01:06 best_checkpoint_run4-2ep-lr1.5e4-r16
drwx------ 2 root root 4.0K Apr 24 01:46 best_checkpoint_run5-2ep-lr2e4-r32
drwx------ 2 root root 4.0K Apr 23 04:44 lora_adapters
drwx------ 2 root root 4.0K Apr 23 04:44 merged_model
-rw------- 1 root root 484K Apr 24 01:46 mlflow_fresh.db
-rw------- 1 root root 158K Apr 18 07:25 test_data.json
-rw------- 1 root root 1.5M Apr 18 07:26 train_data.json
-rw------- 1 root root 1.5M Apr 18 07:25 train_dataset_sft.jsonl
-rw------- 1 root root 156K Apr 18 07:25 val_data.json
-rw------- 1 root root 155K Apr 18 07:25 val_dataset_sft.jsonl

Adapters:
/content/drive/MyDrive/review-summarizer/adapters_run4-2ep-lr1.5e4-r16
/content/drive/MyDrive/review-summarizer/adapters_run5-2ep-lr2e4-r32


In [15]:
# Merge the best model - Merge using Unsloth's own method
# After seeing results, pick the winner
from unsloth import FastLanguageModel

# Reload base model fresh
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=2048, dtype=None, load_in_4bit=True,
)

# Apply LoRA with same config as winning run
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)

# Load the trained adapter weights
adapter_path = f"{save_dir}/adapters_run4-2ep-lr1.5e4-r16"
model.load_adapter(adapter_path, adapter_name="default")
print("Adapters loaded!")

# Try merge with save_method="lora"
model.save_pretrained_merged(
    "merged_model",
    tokenizer,
    save_method="lora",
)
print("Merged!")

==((====))==  Unsloth 2026.4.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Adapters loaded!


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in merged_model/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in merged_model.


...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files:  67%|██████▋   | 2/3 [07:26<03:41, 221.58s/it]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 3/3 [10:18<00:00, 206.28s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [04:51<00:00, 97.33s/it]


Unsloth: Merge process complete. Saved to `/content/merged_model`
Merged!
